# Notebook to process lysosomal and mitochondrial morphology from cellprofiler features 
Inputs Required
 - Database with cellprofiler outputs
   - Mainly use the Per_Cell table for per-cell features
     - Intensity
     - AreaShape features
     - Ratio of organelle area to total cell area
     - Texture features
     - Granularity features
     - Radial intensity distribution about nucleus
   - May additionally want to use mitochdondria or lysosomes.csv if analyzing these specifically 
     - Note that lysosome segmentation is not perfect (but i'm proud of it)
 - Metadata CSV representing 96-well platemap
   - Long-form table containing passage number, staining conditons, treatments, etc
   - Produced automatically from an 8x12 table with the 96Well_PlateMap code
Outputs
- graphs for individual features
- clustering
## Set your paths here

In [2]:
#Imports
import os
import numpy as np
import pandas as pd
import sqlite3
#plotting
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

#import joypy
from scipy import stats

from helpers import *


## Test area for a single tablee

In [3]:
db_path = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/20250501_rep07_output/v2_active/20250501_output.db"
conn = sqlite3.connect(db_path)
cell_df = pd.read_sql_query("SELECT * FROM Per_Cell", conn)
image_df = pd.read_sql_query("SELECT * FROM Per_Image", conn)
cursor = conn.cursor()

combo_df = cell_df.merge(image_df, on=["ImageNumber"], how="left")
combo_df.columns = combo_df.columns.str.replace(r'^Image_Metadata_', 'Metadata_', regex=True)

#display(combo_df[combo_df["Metadata_Well"] == "B01"])
#note the 0328 seems to not be working with the metadata 
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print([table[0] for table in tables])

cursor.execute("PRAGMA table_info(Per_Image);")
columns_info = cursor.fetchall()

# Extract just the column names
column_names = [col[1] for col in columns_info]
print(column_names)

quer = cursor.execute("SELECT * FROM Per_Cell WHERE ImageNumber = ")
last = cursor.fetchall()
display(last)

OperationalError: unable to open database file

### Test metadata

In [ ]:
search_string = "Metadata"

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [row[0] for row in cursor.fetchall()]

for table in tables:
    cursor.execute(f"PRAGMA table_info({table});")
    columns = [col[1] for col in cursor.fetchall()]
    metadata_cols = [col for col in columns if search_string in col]
    if metadata_cols:
        print(f"Table: {table}")
        print("Columns containing 'Metadata':", metadata_cols)
        



In [ ]:
#Debugging functions
def add_drug_to_group(init_df, group, drug):
    '''
        Add the name of a drug treatment from the "Drug" column to the main "group" column
        
        Returns
            Series object: A series containing the column with the drug added to the group
    '''
    if drug is not None:
        # Replace values in 'col1' with values from 'col2' only if 'col2' is not None or NaN
        df = init_df.copy()
        df[group] = np.where(df[drug].notna(), df[drug], df[group])
        newcol = df[group]

    return newcol

#display(cell_df[["Cell_Mean_Nuclei_AreaShape_Area", "Cell_AreaShape_Area"]])

def cell_filters(df):
    not_empty_df = df[df["Metadata_EmptyImage_Cell"] == 0] 
    normal_cells = not_empty_df[not_empty_df["Cell_Classify_Normal"] ==1] #one nucleus only
    size_filtered_cells = normal_cells[df["Cell_AreaShape_Area"] > df["Cell_Mean_Nuclei_AreaShape_Area"]] #cell area bigger than nuclear area
    final_df = size_filtered_cells.reset_index(drop=True)
    return final_df

def multinucleate_cells(df):
    multinuc_df = df[df["Cell_Classify_multinucleate"] ==1]
    return multinuc_df

#cell_df_2 = cell_filters(cell_df)
#cell_df_2.head()
#filter_df = cell_filters(cell_df)
#print(cell_df.shape, " ", filter_df.shape)
#display(filter_df["Cell_Classify_Normal"])

In [ ]:

plate_dfs = []

root = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/20250501_rep07_output/v2_active/"
filename = "20250501_output_extraRow1.db"
db_path = os.path.join(root, filename)
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
plate = "20250501_rep07"
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print([table[0] for table in tables])

cursor.execute("PRAGMA table_info(Per_Image);")
columns_info = cursor.fetchall()

# Extract just the column names
column_names = [col[1] for col in columns_info]
print(column_names)



In [ ]:

plate_dfs = {}
nuclei_dfs = {}

pre_cell_df = pd.read_sql_query("SELECT * FROM Per_Object", conn)
image_df = pd.read_sql_query("SELECT * FROM Per_Image", conn)
cursor = conn.cursor()
#metadata extraction
map_file = os.path.join('plate_metadata', f"{plate}_metadata", "map.csv")
print(map_file)

cell_df = pre_cell_df.merge(image_df, on=["ImageNumber"], how="left")
cell_df.columns = cell_df.columns.str.replace(r'^Image_Metadata_', 'Metadata_', regex=True)
metadata_cols = [col for col in cell_df.columns if "Metadata" in col]
#display(cell_df[metadata_cols])

#make these metadatas string
cell_df['Metadata_WellRow'] = cell_df['Metadata_WellRow'].astype(int)
cell_df['Metadata_WellColumn'] = cell_df['Metadata_WellColumn'].astype(int)
cell_df['Metadata_Field'] = cell_df['Metadata_Field'].astype(int)
display(cell_df[metadata_cols])
cell_df.reset_index(drop=True)

print(os.path.exists(map_file))
if os.path.exists(map_file):
    platemap_df = pd.read_csv(map_file)
    display(platemap_df)
    
    platemap_df['Metadata_WellRow'] = platemap_df['Metadata_WellRow'].astype(int)
    platemap_df['Metadata_WellColumn'] = platemap_df['Metadata_WellColumn'].astype(int)
    platemap_df['Metadata_Field'] = platemap_df['Metadata_Field'].astype(int)
    #platemap_df.reset_index(drop=True)
    cell_df = cell_df.merge(platemap_df, on=['Metadata_WellRow', 'Metadata_WellColumn', 'Metadata_Field'], how='left')
    display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
    cell_df["Passage Group"] = cell_df['PassageNumber'].apply(passage_group)
    cell_df["AllGroups"] = add_drug_to_group(cell_df, "Passage Group", "Drug")
    debug_df = cell_df[["PassageNumber","Passage Group","Drug","AllGroups","Metadata_WellRow","Metadata_WellColumn","Metadata_Field"]]
   
    #display(cell_df[cell_df["Metadata_Well"] == "A02"])
    
    cell_df["Metadata_Plate"] = plate
    cell_df["Replicate_Number"] = plate[-1]
plate_dfs[plate] = cell_df

combined_cell_df = pd.concat(plate_dfs.values(), ignore_index=True)


In [ ]:



root = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/20250410_rep06_output/v2_active/"
filename = "20250410_output.db"
db_path = os.path.join(root, filename)
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
plate = "20250410_rep06"

plate_dfs = {}
nuclei_dfs = {}

pre_cell_df = pd.read_sql_query("SELECT * FROM Per_Cell", conn)
image_df = pd.read_sql_query("SELECT * FROM Per_Image", conn)
#metadata extraction
map_file = os.path.join('plate_metadata', f"{plate}_metadata", "map.csv")
print(map_file)

cell_df = pre_cell_df.merge(image_df, on=["ImageNumber"], how="left")
cell_df.columns = cell_df.columns.str.replace(r'^Image_Metadata_', 'Metadata_', regex=True)
metadata_cols = [col for col in cell_df.columns if "Metadata" in col]
#display(cell_df[metadata_cols])

#make these metadatas string
cell_df['Metadata_WellRow'] = cell_df['Metadata_WellRow'].astype(int)
cell_df['Metadata_WellColumn'] = cell_df['Metadata_WellColumn'].astype(int)
cell_df['Metadata_Field'] = cell_df['Metadata_Field'].astype(int)
display(cell_df[metadata_cols])
cell_df.reset_index(drop=True)

print(os.path.exists(map_file))
if os.path.exists(map_file):
    platemap_df = pd.read_csv(map_file)
    display(platemap_df)
    
    platemap_df['Metadata_WellRow'] = platemap_df['Metadata_WellRow'].astype(int)
    platemap_df['Metadata_WellColumn'] = platemap_df['Metadata_WellColumn'].astype(int)
    platemap_df['Metadata_Field'] = platemap_df['Metadata_Field'].astype(int)
    #platemap_df.reset_index(drop=True)
    cell_df = cell_df.merge(platemap_df, on=['Metadata_WellRow', 'Metadata_WellColumn', 'Metadata_Field'], how='left')
    display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
    cell_df["Passage Group"] = cell_df['PassageNumber'].apply(passage_group)
    cell_df["AllGroups"] = add_drug_to_group(cell_df, "Passage Group", "Drug")
    debug_df = cell_df[["PassageNumber","Passage Group","Drug","AllGroups","Metadata_WellRow","Metadata_WellColumn","Metadata_Field"]]
   
    #display(cell_df[cell_df["Metadata_Well"] == "A02"])
    
    cell_df["Metadata_Plate"] = plate
    cell_df["Replicate_Number"] = plate[-1]
plate_dfs[plate] = cell_df

combined_cell_df = pd.concat(plate_dfs.values(), ignore_index=True)
#display(combined_cell_df)

## Actually loop over plates to check things

In [ ]:
# Setting file paths
curr_plates = ["20240313_rep01","20240326_rep02","20241018_rep03", "20241112_rep04", "20250328_rep05", "20250410_rep06", "20250501_rep07"] #"20240313_rep01_output","20240326_rep02_output"
parent_dir = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output"

table_names = ['Per_Cell', 'Per_Nuclei','Per_MergedMitoPerCell','Per_MergedLysoPerCell']

# Initialize a list to store the combined DataFrames
plate_dfs = {}
nuclei_dfs = {}

# Loop over the plates
for root, dirs, files in os.walk(parent_dir):
    for filename in files: 
        if filename.endswith(".db") and "active" in root and "extraRow1" not in filename:
            db_path = os.path.join(root, filename) #make the path
            for plate in curr_plates:
                if plate in db_path:
                    conn = sqlite3.connect(db_path)
                    cursor = conn.cursor()
                    try:
                        # Read the 'Per_Cell' table and get metadata from 'Per_Image' table
                        pre_cell_df = pd.read_sql_query("SELECT * FROM Per_Cell", conn)
                        image_df = pd.read_sql_query("SELECT * FROM Per_Image", conn)
                        #metadata extraction - make sure its the exact same file format as the one above
                        map_file = os.path.join('plate_metadata', f"{plate}_metadata", "map.csv")
                        print(map_file)
                        
                        cell_df = pre_cell_df.merge(image_df, on=["ImageNumber"], how="left")
                        cell_df.columns = cell_df.columns.str.replace(r'^Image_Metadata_', 'Metadata_', regex=True)
                        
                        #make these metadatas string
                        cell_df['Metadata_WellRow'] = cell_df['Metadata_WellRow'].astype(int)
                        cell_df['Metadata_WellColumn'] = cell_df['Metadata_WellColumn'].astype(int)
                        cell_df['Metadata_Field'] = cell_df['Metadata_Field'].astype(int)
                        
                        cell_df.reset_index(drop=True)
                        
                        print(os.path.exists(map_file))
                        
                        if os.path.exists(map_file):
                            platemap_df = pd.read_csv(map_file)
                            display(platemap_df)
                            platemap_df['Metadata_WellRow'] = platemap_df['Metadata_WellRow'].astype(int)
                            platemap_df['Metadata_WellColumn'] = platemap_df['Metadata_WellColumn'].astype(int)
                            platemap_df['Metadata_Field'] = platemap_df['Metadata_Field'].astype(int)
                            #platemap_df.reset_index(drop=True)
                            cell_df = cell_df.merge(platemap_df, on=['Metadata_WellRow', 'Metadata_WellColumn', 'Metadata_Field'], how='left')
                            #display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
                            cell_df["Passage Group"] = cell_df['PassageNumber'].apply(passage_group)
                            cell_df["AllGroups"] = add_drug_to_group(cell_df, "Passage Group", "Drug")
                            debug_df = cell_df[["PassageNumber","Passage Group","Drug","AllGroups","Metadata_WellRow","Metadata_WellColumn","Metadata_Field"]]
                        
                            display(debug_df.head(10))
                            
                            cell_df["Metadata_Plate"] = plate
                            cell_df["Replicate_Number"] = plate[-1]
                            
                        plate_dfs[plate] = cell_df
                    except Exception as e:
                        print(f"Error reading {db_path}: {e}")
                    finally:
                        conn.close()

# Combine all DataFrames
combined_cell_df = pd.concat(plate_dfs.values(), ignore_index=True)
#combined_nuclei_df = pd.concat(nuclei_dfs.values(), ignore_index=True)
                

#Filter DataFrames to only include cells that were stained with LAMP1-488 and MitoRed

#combined_nuclei_df_mitolyso = combined_nuclei_df[combined_nuclei_df['Staining'].str.startswith("LAMP1-488 + MitoRed")]



In [ ]:
def well_namer(row, col):
    '''
        Convert row and column numbers to a well name in the format A01, B02, etc.
        
        Args:
            row (int): The row number (1-8)
            col (int): The column number (1-12)
        
        Returns:
            str: Well name in the format A01, B02, etc.
    '''
    well_name = str(chr(ord('@')+ row)) + str(col).rjust(2, '0')  #make the number have a left align, adding a zero
    return well_name

def add_well_metadata(image_df):
    '''
        Add well metadata to the image DataFrame.
        
        Args:
            image_df (DataFrame): DataFrame containing image metadata
        
        Returns:
            DataFrame: Updated DataFrame with well metadata
    '''
    image_df.columns = image_df.columns.str.replace(r'^Image_Metadata_', 'Metadata_', regex=True)
    image_df[["Metadata_WellRow","Metadata_WellColumn","Metadata_Field"]] = image_df["Image_URL_DAPI"].str.extract(r'r(\d{2})c(\d{2})f(\d{2}).tif')
    # Convert extracted columns to int
    image_df["Metadata_WellRow"] = image_df["Metadata_WellRow"].astype(int)
    image_df["Metadata_WellColumn"] = image_df["Metadata_WellColumn"].astype(int)
    image_df["Metadata_Field"] = image_df["Metadata_Field"].astype(int)
    # apply well namer function
    image_df["Metadata_Well"] = image_df.apply(lambda x: well_namer(x["Metadata_WellRow"], x["Metadata_WellColumn"]), axis=1)
    display(image_df[["ImageNumber","Image_URL_DAPI","Metadata_WellRow","Metadata_WellColumn","Metadata_Field","Metadata_Well"]])
    
    return image_df

def update_database_with_well_metadata(db_path):
    '''
        Update the database with well metadata.
        
        Args:
            db_path (str): Path to the database file
    '''
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Read Per_Image table
    image_df = pd.read_sql_query("SELECT * FROM Per_Image", conn)
    
    # Add well metadata
    updated_image_df = add_well_metadata(image_df)
    
    # Write updated DataFrame back to the database
    try:
        updated_image_df.to_sql('Per_Image', conn, if_exists='replace', index=False)
        print("Database updated successfully with well metadata.")
    except Exception as e:
        print(f"Error updating database: {e}")
    #cursor.execute("SELECT Metadata_Well FROM Per_Image LIMIT 5;")
    #cursor.fetchall()
    conn.close()





In [ ]:
#Fix the missing metadata in one of the plates
root = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/20250328_rep05_output/v2_active/"
filename = "20250328_output.db"
db_path = os.path.join(root, filename)
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
plate = "20250328_rep05"

plate_dfs = {}
nuclei_dfs = {}

pre_cell_df = pd.read_sql_query("SELECT * FROM Per_Cell", conn)
image_df = pd.read_sql_query("SELECT * FROM Per_Image", conn)
image_df = add_well_metadata(image_df)

#metadata extraction
map_file = os.path.join('plate_metadata', f"{plate}_metadata", "map.csv")
print(map_file)

cell_df = pre_cell_df.merge(image_df, on=["ImageNumber"], how="left")
cell_df.columns = cell_df.columns.str.replace(r'^Image_Metadata_', 'Metadata_', regex=True)
metadata_cols = [col for col in cell_df.columns if "Metadata" in col]
#display(cell_df[metadata_cols])

#make these metadatas string
cell_df['Metadata_WellRow'] = cell_df['Metadata_WellRow'].astype(int)
cell_df['Metadata_WellColumn'] = cell_df['Metadata_WellColumn'].astype(int)
cell_df['Metadata_Field'] = cell_df['Metadata_Field'].astype(int)
display(cell_df[metadata_cols])
cell_df.reset_index(drop=True)

print(os.path.exists(map_file))
if os.path.exists(map_file):
    platemap_df = pd.read_csv(map_file)
    display(platemap_df)
    
    platemap_df['Metadata_WellRow'] = platemap_df['Metadata_WellRow'].astype(int)
    platemap_df['Metadata_WellColumn'] = platemap_df['Metadata_WellColumn'].astype(int)
    platemap_df['Metadata_Field'] = platemap_df['Metadata_Field'].astype(int)
    #platemap_df.reset_index(drop=True)
    cell_df = cell_df.merge(platemap_df, on=['Metadata_WellRow', 'Metadata_WellColumn', 'Metadata_Field'], how='left')
    display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
    cell_df["Passage Group"] = cell_df['PassageNumber'].apply(passage_group)
    cell_df["AllGroups"] = add_drug_to_group(cell_df, "Passage Group", "Drug")
    debug_df = cell_df[["PassageNumber","Passage Group","Drug","AllGroups","Metadata_WellRow","Metadata_WellColumn","Metadata_Field"]]
   
    #display(cell_df[cell_df["Metadata_Well"] == "A02"])
    
    cell_df["Metadata_Plate"] = plate
    cell_df["Replicate_Number"] = plate[-1]

plate_dfs[plate] = cell_df

In [ ]:
#Export to a giant csv
combined_cell_df_mitolyso = combined_cell_df[combined_cell_df['Staining'].str.startswith("LAMP1-488 + MitoRed")]
#filter_df = cell_filters(combined_cell_df_mitolyso)
display(combined_cell_df_mitolyso.head(10))
#print(cell_df.shape, " ", filter_df.shape)
outpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/postprocessed_csvs"
combined_cell_df_mitolyso.to_csv(os.path.join(outpath, "total_combined_cell.csv"), index=False)

In [ ]:
display(combined_cell_df)
combined_cell_df_mitolyso = combined_cell_df[combined_cell_df['Staining'].str.startswith("LAMP1-488 + MitoRed")]
#combined_cell_df_mitolyso_merged = plate_df_setup(curr_plates, curr_plate_datafolders, parent_dir, ['Cell.csv', 'Nuclei.csv','MergedMitoPerCell.csv','MergedLysoPerCell.csv'])
display(combined_cell_df_mitolyso)
#display(combined_cell_df_mitolyso_merged)

In [ ]:
display(combined_cell_df_mitolyso[['AreaShape_Area', 'Children_Mitochondria_Count','Mean_Mitochondria_AreaShape_Area']])

### Extra cell to export the combined DataFrames to CSV files

In [ ]:

out_dir = os.path.join(parent_dir, 'export')
for plate, df in plate_dfs.items():
    plate_out_dir = os.path.join(out_dir, plate)
    #os.makedirs(plate_out_dir, exist_ok=True)
    #df.to_csv(os.path.join(plate_out_dir, 'Cell_w_metadata.csv'), index=False)
#combined_cell_df.to_csv(os.path.join(out_dir, 'All_Cell_w_metadata.csv'), index=False)


# Define the cell features


In [1]:
#Add extra columns 
def proportion_area_occupied_per_cell(df, compartment):
    #proportion of area occupied = children * mean organelle area / cell area
    colname = 'Total_Area_Proportion_' + compartment + '_Per_Cell'
    
    #children = 'Children_' + compartment + '_Count'
    #mean_organelle_area = 'Mean_'+ compartment + '_AreaShape_Area'
    organelle_area = compartment + '_AreaShape_Area'
    cell_area = 'AreaShape_Area'
    #df[colname] = df.apply(lambda x: (x[children] * x[mean_organelle_area]) / x[cell_area], axis=1)
    df[colname] = df.apply(lambda x: (x[organelle_area]) / x[cell_area], axis=1)
    return df[colname]

def mean_intesity_per_compartment_per_cell(df,compartment, name, tag):
    # Calculate the mean intensity of each compartment per cell
    #mean_intesity_per_compartment = integrated / (children*mean_area)
    colname = 'MeanIntensity_Per_' + compartment + '_Per_Cell'
    integrated = 'Intensity_IntegratedIntensity_' + tag
    #children = 'Children_' + compartment + '_Count'
    #mean_area = 'Mean_'+ compartment + '_AreaShape_Area'
    total_organelle_area = name + '_AreaShape_Area'
    df[colname] = df.apply(lambda x: x[integrated] / x[total_organelle_area], axis=1)
    return df[colname]


combined_cell_df_mitolyso_merged["Mean_Mitochondria_Area_PerCell_Ratio"]  = proportion_area_occupied_per_cell(combined_cell_df_mitolyso_merged, "MergedMitoPerCell")
combined_cell_df_mitolyso_merged["Mean_Lysosomes_Area_PerCell_Ratio"]  = proportion_area_occupied_per_cell(combined_cell_df_mitolyso_merged, "MergedLysoPerCell")

combined_cell_df_mitolyso_merged["MeanIntensity_Lysosomes_PerCell_Ratio"] = mean_intesity_per_compartment_per_cell(combined_cell_df_mitolyso_merged, "Lysosomes", "MergedLysoPerCell", "LAMP1")
combined_cell_df_mitolyso_merged["MeanIntensity_Mitochondria_PerCell_Ratio"] = mean_intesity_per_compartment_per_cell(combined_cell_df_mitolyso_merged, "Mitochondria", "MergedMitoPerCell", "MitoTracker")


display(combined_cell_df_mitolyso_merged[["Passage Group","Mean_Lysosomes_Intensity_MeanIntensity_LAMP1","MeanIntensity_Mitochondria_PerCell_Ratio","MeanIntensity_Lysosomes_PerCell_Ratio","Mean_Lysosomes_Area_PerCell_Ratio","Mean_Mitochondria_Area_PerCell_Ratio","Math_Total_Mitochondria_AreaShape_Area_PerCell","Math_Total_Lysosomes_AreaShape_Area_PerCell"]])

NameError: name 'combined_cell_df_mitolyso_merged' is not defined

In [ ]:

#file_path = '/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/Cellcsv_columns.txt'
pairs_samples = [('P6-8', 'P9-10'), ('P6-8', 'P11-13'), ('P6-8', 'P14-16'), ('P6-8', 'P17-18'), ('P6-8', 'P20-21'), ('P6-8', 'P22-24'), ('P9-10', 'P11-13'), ('P9-10', 'P14-16'), ('P9-10', 'P17-18'), ('P9-10', 'P20-21'), ('P9-10', 'P22-24'), ('P11-13', 'P14-16'), ('P11-13', 'P17-18'), ('P11-13', 'P20-21'), ('P11-13', 'P22-24'), ('P14-16', 'P17-18'), ('P14-16', 'P20-21'), ('P14-16', 'P22-24'), ('P17-18', 'P20-21'), ('P17-18', 'P22-24'), ('P20-21', 'P22-24')]


columns_list = define_cell_features(combined_cell_df_mitolyso_merged)

mito_features = make_feature_dict([col for col in columns_list if 'Mito' in col])
lyso_features = make_feature_dict([col for col in columns_list if 'Lysosome' in col or 'LAMP1' in col or 'Lyso' in col])
nuc_features = make_feature_dict([col for col in columns_list if 'Nuc' in col or 'DAPI' in col])
print(columns_list)


## Feature lists here:

In [ ]:
feature_dicts = [mito_features, lyso_features, nuc_features]
feature_names = ["Mitochondria Features", "Lysosome Features", "Nucleus Features"]

# Define the output file path
output_file_path = 'allfeatures_file.md'

# Open the file in write mode
with open(output_file_path, 'w') as file:
    for feature_name, feature_dict in zip(feature_names, feature_dicts):
        file.write(f"# {feature_name}\n\n")
        for feature_type, features in feature_dict.items():
            file.write(f"## {feature_type.capitalize()}\n")
            for feature in features:
                file.write(f"- {feature}\n")
            file.write("\n")
            
print(f"List has been written to {output_file_path}")


In [ ]:
combined_cell_df_mitolyso_merged.head()

sns.pairplot(cell_df, hue='Passage Group', vars=mito_features['radialdistribution'], diag_kind='kde', plot_kws={'alpha':0.5})
plt.show()

# Functions for Data Analysis
calulcate normalizations, remove extreme left outliers, etc

## Normalize features to control (Passage 6-8)

In [ ]:
norm_cell_df = combined_cell_df_mitolyso_merged.copy()
norm_cell_df_nuc = apply_feature_normalization(norm_cell_df, nuc_features, curr_plates).copy()
norm_cell_df_mito = apply_feature_normalization(norm_cell_df_nuc, mito_features, curr_plates).copy()
norm_cell_df_mitolyso = apply_feature_normalization(norm_cell_df_mito, lyso_features, curr_plates)

#watch out - merged df might be clipping off all of the features


In [ ]:
#norm_cell_df_mitolyso.loc[norm_cell_df_mitolyso['Passage Group'] == 'P6-8'].describe()
norm_cell_df_mitolyso.describe()


## Trying out the pivot and groupby functions


In [ ]:
#average_grouped_df = average_groups_by_plate(norm_cell_df_mitolyso, x_value='Passage Group', y_value='Intensity_MeanIntensity_MitoTracker', replicates='Replicate_Number')
#average_grouped_df_pivot = average_groups_pivot(average_grouped_df, x_value='Passage Group', y_value='Intensity_MeanIntensity_MitoTracker', replicates='Replicate_Number')
#display(average_grouped_df)
#display(average_grouped_df_pivot)


In [ ]:
categorical_col = 'Passage Group'
order = ['P6-8', 'P9-10', 'P11-13', 'P14-16', 'P17-18', 'P20-21', 'P22-24']
df_test = norm_cell_df_mitolyso.copy()

pairs = getpairs(df_test, categorical_col, order)

# Print the pairs
print(pairs)

In [ ]:
def old_stats(df, cols, excel_name):
    stats_cols = cols
    with pd.ExcelWriter(excel_name) as writer:

      for column in cols:
          temp_copy = df.copy()  # Create a copy of the DataFrame for processing

          #temp_copy = outlier_removal(temp_copy, column)
          tukey = pairwise_tukeyhsd(endog=temp_copy[column], groups=temp_copy['Passage Group'], alpha=0.05)

        # Extract relevant results
          results = np.array(tukey.summary().data)[:, [0, 1, 3, 6]]
          df_results = pd.DataFrame(results, columns=['Group 1', 'Group 2', 'p-value', 'Reject']).drop([0])
          df_results.reset_index(drop=True, inplace=True)
          df_results[['Group 1', 'Group 2']] = df_results[['Group 1', 'Group 2']].astype(int)
          df_results['p-value'] = df_results['p-value'].astype(float)
        # Truncate the column name if it exceeds 31 characters
          name = column[:31]
          

        # Save DataFrame to Excel sheet without the index column
          df_results.to_excel(writer, sheet_name=name, index=False)


In [ ]:
def tukey_test_1(data, test_groups, feature):
  '''
  Perform a oneway anova test and a pairwise tukey post hoc test using averaged values per replicate
  Returns a dataframe
  '''
  from scipy.stats import f_oneway
  from statsmodels.stats.multicomp import pairwise_tukeyhsd
  
  df = data.copy()
  
  #groups = getpairs(temp_copy, 'Passage Group')
  #calculate tukey HSD
  
  tukey = pairwise_tukeyhsd(endog=df[feature], groups=df[test_groups], alpha=0.05)

  # Extract relevant results
  results = np.array(tukey.summary().data)[:, [0, 1, 3, 6]]
  df_results = pd.DataFrame(results, columns=['Group 1', 'Group 2', 'p-value', 'Reject']).drop([0])
  df_results.reset_index(drop=True, inplace=True)
  df_results[['Group 1', 'Group 2']] = df_results[['Group 1', 'Group 2']]
  df_results['p-value'] = df_results['p-value'].astype(float)
  
  return df_results
  # Display DataFrame inline
  #display(df_results)




In [ ]:
feature_meas = "Mean_Mitochondria_AreaShape_MajorAxisLength"

display(make_single_feature_df(norm_cell_df_mitolyso, 'Passage Group', feature_meas, 'Replicate_Number'))


## It's plotting time


In [ ]:
order = ['P6-8', 'P9-10', 'P11-13', 'P14-16', 'P17-18', 'P20-21', 'P22-24']

def average_groups_by_plate_1(df, x_value, y_value, replicates):
    '''
    Group the DataFrame by the specified columns and calculate the mean of the y_value column.
    Returns the averaged dataframe for plotting
    '''
    df = df.dropna(subset=[x_value, y_value, replicates])
    df = df[df[y_value] != 0]

    df.reset_index(drop=True, inplace=True)
    
    group_averages = df.groupby([x_value, replicates], as_index=False, observed=True).agg({y_value: "mean"})
    
    # Reset the index to get a clean DataFrame
    average_df = group_averages.reset_index()
   
    return average_df

def make_single_feature_df_1(data, group, feature, replicates):
  pd.options.mode.copy_on_write = True
  
  subset=[group, feature, replicates]
  
  df = data.dropna(subset = subset).reset_index(drop=True)
  df = df[df[feature] != 0]
  
  df_subset = df[subset]
  df_subset[group] = df[group].astype('category')
  df_subset.reset_index(drop = True, inplace = True)
  
  return df_subset 

def oneway_anova(data, group_name, feature_meas):
  from scipy.stats import f_oneway

  data = data.dropna(subset=[group_name, feature_meas])
  data = data[data[feature_meas] != 0]
  
  groups = data[group_name].unique()
  data = [data[data[group_name] == group][feature_meas].dropna() for group in groups]
  anova_result = f_oneway(*data)
  
  print(f"ANOVA F-statistic: {anova_result.statistic}, ANOVA p-value: {anova_result.pvalue}")
  return anova_result


In [ ]:
          
feature_meas = 'Children_Lysosomes_Count'
pairs = getpairs(norm_cell_df_mitolyso, 'Passage Group', order)

feature_df = make_single_feature_df_1(norm_cell_df_mitolyso, group='Passage Group', feature=feature_meas, replicates='Replicate_Number')
group_avg_df = average_groups_by_plate_1(feature_df, x_value='Passage Group', y_value=feature_meas, replicates='Replicate_Number')
group_avg_df_pivot = average_groups_pivot(group_avg_df, x_value='Passage Group', y_value=feature_meas, replicates='Replicate_Number')

display(feature_df)
display(group_avg_df)
display(group_avg_df_pivot)

sns.set_theme(style="ticks")
#sns.set_context("notebook", font_scale=1.9)

plt.figure(figsize=(12, 8))

sns.violinplot(data=feature_df, x='Passage Group',
            y=feature_meas,
            hue = "Passage Group",
            order=order,
            fill = False,
            palette = 'pastel',
            cut=0,
            linecolor= 'k',
            inner_kws=dict(box_width = 5))

ax = sns.swarmplot(data=group_avg_df, x='Passage Group',
            y=feature_meas,
            hue = "Replicate_Number",
            order=order,
            palette='Set2',
            size=10, 
            edgecolor="k", 
            linewidth=1,
            dodge=False)

ax.legend_.remove()

sns.despine()
plt.gcf()#.set_size_inches(10, 6)
plt.xlabel('Passage Group')
plt.ylabel(feature_meas.replace('_',' '))

from statannotations.Annotator import Annotator
from statannotations.stats.StatTest import StatTest

# Extract the data for each group

# Perform the one-way ANOVA test

# Print the results
anova = oneway_anova(group_avg_df, 'Passage Group', feature_meas)

tukey_results = tukey_test_1(group_avg_df, test_groups='Passage Group', feature=feature_meas)
display(tukey_results)

annotator = Annotator(ax, pairs, data=group_avg_df_pivot, order=order)
annotator.configure(text_format='star', loc='inside', verbose = 2)
annotator.set_pvalues_and_annotate(tukey_results['p-value'])

#plt.savefig(feature_meas + '_superviolinplot.png', dpi=300)
plt.show()



#### For Kruskal-Walis
```python
sns.catplot(data=feature_df, x='Passage Group',
            y=feature_meas,
            hue = "Replicate_Number",
            order=order,
            fill = False,
            palette='Set2',
            kind = 'violin',
            inner = None)

sns.boxplot(data=feature_df, x='Passage Group',
            y=feature_meas,
            hue = "Replicate_Number",
            order=order,
            showfliers = False,
            palette='Set2')

ax = sns.swarmplot(data=group_avg_df, x='Passage Group',
            y=feature_meas,
            hue = "Replicate_Number",
            order=order)

sns.despine()
plt.gcf().set_size_inches(10, 6)
plt.xlabel('Passage Group')
plt.ylabel(feature_meas.replace('_',' '))
plt.ylim(-0.5, 6)


from statannotations.Annotator import Annotator
from statannotations.stats.StatTest import StatTest

annotator = Annotator(ax, pairs, data=group_avg_df_pivot, x='Passage Group', y=feature_meas, order=order) 0 
annotator.configure(test='Kruskal', text_format='star', loc='inside')
annotator.apply_and_annotate()


plt.savefig(feature_meas + '_boxplot.png', dpi=300)
```


## Make the plots - nonparametric
### ytitles
ytitle = "Distance of Mitochondria from Cell Center"
ytitle = "Distance of Lysosomes from Cell Center"
"Number of Mitochondria per Cell (Relative to Control)"
ytitle = "Number of Lysosomes per Cell (Relative to Control)"
ytitle = "Total Normalized Mitochondrial Area Per Cell"
ytitle = "Normalized # of Mitochondrial Endpoints
ytitle = "Normalized # of Mitochondrial Trunks"
ytitle = "Mean Length of Mitochondrial Network"
ytitle = "Mean # of Mitochondrial Branches"
ytitle = "Minimum Distance of Mitochondria from Cell"
ytitle = "Mean Total Mitochondrial Area"
ytitle = "Mean Mitochondrial Size"
ytitle = "Mean Total Mitochondrial Perimeter"
ytitle = "Mean Mitochondrial Granularity"
ytitle = "Total Mitochondrial Intensity"
ytitle = "Mean LAMP1 Intensity Per Lysosome"
ytitle = "Mean Mitochondria Intensity Per Cell"
ytitle = "Mean Lysosome Intensity Per Cell"
ytitle = "Total LAMP1 Intensity"
ytitle = "Total Mitochondrial Intensity"
ytitle = "Mean MitoTracker Intensity"
ytitle = "Ratio of Mitochondria Area to Cell Area"
ytitle = "Ratio of Lysosome Area to Cell Area"
ytitle = "MitoTracker Intensity Per Cell Area"
ytitle = "LAMP1 Intensity Per Cell Area"
ytitle = "Lysosomal Circularity"
ytitle = "MitoTracker Intensity Per Cell Area"

In [ ]:
#Nonparametric Function version
feature_meas = 'Mean_Lysosomes_Area_PerCell_Ratio'
order = ['P6-8', 'P9-10', 'P11-13', 'P14-16', 'P17-18', 'P20-21']#, 'P22-24']

if feature_meas == 'AreaShape_Area':
    norm_cell_df_mitolyso[feature_meas] = normalize_to_control(norm_cell_df_mitolyso,feature_meas)
    ytitle = "Mean Cell Size"


ytitle = "Mean Lysosome Area Per Cell Area"

ylimit = None# (-1,2)

pallete = "deep"
#pallete = "pastel" 
remove_outliers = True

def remove_outliers_iqr(df, col = None):
    cols = df.select_dtypes('number').columns  # limits to a (float), b (int) and e (timedelta)
    df_sub = df.loc[:, cols]

    iqr = df_sub.quantile(0.75, numeric_only=False) - df_sub.quantile(0.25, numeric_only=False)
    
    #calculate  extreme outlisers by dividing median by iqr
    lim = np.abs((df_sub - df_sub.median()) / iqr) < 2.22

    # replace outliers with nan
    df.loc[:, cols] = df_sub.where(lim, np.nan)
    df.dropna(subset=cols, inplace=True) # drop rows with NaN in numerical columns
    return df

def make_superplot_with_kruskal(data, group, feature_meas, replicates, ytitle = None, pallete='pastel', ylim = None, remove_outliers = remove_outliers):
    
    order = ['P6-8', 'P9-10', 'P11-13', 'P14-16', 'P17-18', 'P20-21']#, 'P22-24']
    
    if ytitle is None:
        ytitle = feature_meas.replace('_', ' ')

       
    feature_df = make_single_feature_df(data, group=group, feature=feature_meas, replicates=replicates)
     
    if remove_outliers is True:
        feature_df = remove_outliers_iqr(feature_df)
        display(feature_df)
        
    
    pairs = getpairs(feature_df, group, order)

    #Remove the n=1 replicate
    feature_df = feature_df[feature_df[group] != "P22-24"]


    group_avg_df = average_groups_by_plate(feature_df, x_value=group, y_value=feature_meas, replicates=replicates)
    group_avg_df_pivot = average_groups_pivot(group_avg_df, x_value=group, y_value=feature_meas, replicates=replicates)

    sns.set_theme(style="ticks")
    sns.set_context("talk", font_scale=0.5)

    plt.figure(dpi=300)

    sns.violinplot(data=feature_df, x=group,
                y=feature_meas,
                order=order,
                fill = False,
                color= 'gainsboro',
                cut=1,
                native_scale=True,
                linecolor='k',
                inner= None,
                #inner_kws=dict(box_width = 5)
                )

    ax = sns.swarmplot(data=group_avg_df, x=group,
                y=feature_meas,
                hue = replicates,
                order=order,
                palette=pallete,
                size=10, 
                edgecolor="k", 
                linewidth=1,
                dodge=0.5)
    
    #use a boxplot to draw the mean line - thinking outside the box :)
    sns.boxplot(data = group_avg_df, x = group,
                y = feature_meas,
                showmeans=True,
                meanline=True,
                meanprops={'color': 'dimgray', 'ls': '-', 'lw': 2.5},
                medianprops={'visible': False},
                whiskerprops={'visible': False},
                zorder=1,
                showfliers=False,
                showbox=False,
                showcaps=False,
                ax = ax)

    ax.legend_.remove()

    sns.despine()
    plt.gcf()#.set_size_inches(10, 6)
    plt.xlabel(group)
    plt.ylabel(ytitle)
    plt.ylim(ylim)

    from statannotations.Annotator import Annotator
    annotator = Annotator(ax, pairs, data=group_avg_df_pivot, order=order) 
    annotator.configure(test='Kruskal', 
                        text_format='star', 
                        loc='inside', 
                        hide_non_significant = True,
                        color = 'black',
                        verbose = 2)
    annotator.apply_and_annotate()

    plt.savefig(feature_meas + '_superviolinplot.png', dpi=300)
    plt.show()
    


make_superplot_with_kruskal(norm_cell_df_mitolyso, 'Passage Group', feature_meas, 'Replicate_Number', ytitle, pallete, ylimit)

In [ ]:
#Nonparametric version
feature_meas = 'Children_Lysosomes_Count'
order = ['P6-8', 'P9-10', 'P11-13', 'P14-16', 'P17-18', 'P20-21']#, 'P22-24']

feature_df = make_single_feature_df_1(norm_cell_df_mitolyso, group='Passage Group', feature=feature_meas, replicates='Replicate_Number')
pairs = getpairs(feature_df, 'Passage Group', order)

#Remove the n=1 replicate
feature_df = feature_df[feature_df['Passage Group'] != "P22-24"]

group_avg_df = average_groups_by_plate_1(feature_df, x_value='Passage Group', y_value=feature_meas, replicates='Replicate_Number')
group_avg_df_pivot = average_groups_pivot(group_avg_df, x_value='Passage Group', y_value=feature_meas, replicates='Replicate_Number')

sns.set_theme(style="ticks")
sns.set_context("talk", font_scale=0.7)

plt.figure(dpi=300)

sns.violinplot(data=feature_df, x='Passage Group',
            y=feature_meas,
            #hue = "Passage Group",
            order=order,
            fill = False,
            color= 'gainsboro',
            #palette = 'Set2',
            cut=2,
            native_scale=True,
            linecolor='k',
            inner= None,
            #inner_kws=dict(box_width = 5)
            )

ax = sns.swarmplot(data=group_avg_df, x='Passage Group',
            y=feature_meas,
            hue = "Replicate_Number",
            order=order,
            palette='pastel',
            size=10, 
            edgecolor="k", 
            linewidth=1,
            dodge=0.5)

sns.pointplot(data=group_avg_df, x='Passage Group',
              y=feature_meas,
              #hue='Replicate_Number',
              color='dimgray',
              order=order,
              dodge=False,
              markers='_',
              linestyle=None,
              errorbar=None,
              ax=ax)

ax.legend_.remove()

sns.despine()
plt.gcf()#.set_size_inches(10, 6)
plt.xlabel('Passage Group')
plt.ylabel(feature_meas.replace('_',' '))
plt.ylim(-1,11)

from statannotations.Annotator import Annotator
annotator = Annotator(ax, pairs, data=group_avg_df_pivot, order=order) 
annotator.configure(test='Kruskal', 
                    text_format='star', 
                    loc='inside', 
                    hide_non_significant = True,
                    color = 'black',
                    verbose = 2)
annotator.apply_and_annotate()

plt.savefig(feature_meas + '_superviolinplot.png', dpi=300)
plt.show()

### To export the normalized csv:


In [ ]:

norm_cell_df_mitolyso.to_csv(os.path.join('All_Cell_w_metadata_normalized.csv'), index=False)

# Make the plots and validate dist

In [ ]:
def make_layout(xtitle,ytitle):
  design = go.Layout(
        plot_bgcolor="#FFF",
        xaxis=dict(
            title=xtitle,
            linecolor="black",
            showgrid=False,
            titlefont=dict(size=20),
            tickfont=dict(size=16, color="black")
        ),
        yaxis=dict(
            title=ytitle,
            linecolor="black",
            showgrid=False,
            titlefont=dict(size=20),
            tickfont=dict(size=16, color="black")
        ),
        font=dict(size=14),
        legend=dict(
            title="",
            itemsizing='constant',
            font=dict(size=16, color="black"),
            tracegroupgap=10,
            traceorder='normal',
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        boxgap=0.4,
        boxgroupgap=0.05,
        width=900,  # Specify width of the plot
        height=600
    )
  return design

def box_param(fig, color, design):
  fig.update_traces(boxmean=True)
  fig.update_traces(jitter=1.0)
  fig.update_traces(boxpoints=False)
  fig.update_layout(design)
  fig.update_traces({'opacity': 0.9})
  fig.update_traces(marker_color=color)
  return fig

In [ ]:
#define parameter for plotly boxplot
def boxplot(df, variable, ytitle, xtitle, box_color, save=False, save_name=None):
    
    # #some styling stuff
  layout = make_layout(xtitle,ytitle)
  temp_copy = df.copy()
  temp_copy = outlier_removal(temp_copy, variable)
  temp_copy2 = normalize_to_control(temp_copy, variable)

  fig = px.box(temp_copy2, x="Passage Group", y=variable)
  fig = box_param(fig, box_color, layout)
  fig.update_traces(quartilemethod="inclusive")
  fig.show()
  
  if save==True:
      fig.write_image(save_name)

##  Call plotly
```python
boxplot(df, # dataframe name: mito_df, nuclei_df, image_df, outline_df, lysosomes_df
        'Variable', # the variable you wanna plot, column name
        'Y Title',# name of your y-axis (custom)
        'Time Point',#name of your x-axis (custom)
        'Red', # color of the boxes
        save=True, # if wanna save change to False to True, False is default
        save_name='_boxplot.png') #specify the name of the plot that you save

```

In [ ]:
#testing
pd.options.mode.copy_on_write = True
boxplot(cell_df,
            'Texture_Contrast_MitoTracker_3_01_256',
            'Texture_Contrast_MitoTracker_3_01_256',
            "Time Point",
            'Red',
            save = False,
            save_name = "_boxplot.png")

In [ ]:
#save all the files for that one feature

colors_i = 0
compartment = 'Mito'
#import kaleido

def save_all_single_feature_plots(data, features):
    '''
    Go through the data given and make plots
    Return: nothing; just outputs the plots
    '''
    for feature in features:
        if colors_i >= len(px.colors.qualitative.Dark24):
            colors_i = 0
        boxplot(data,
                feature,
                feature.replace('_', ' '),
                "Time Point",
                px.colors.qualitative.Dark24[colors_i],
                save = True,
                save_name = plate + "_" + compartment + "_" + feature + "_boxplot.png")
        colors_i = colors_i + 1
    
            
    

## The clustering zone

In [ ]:
from sklearn.decomposition import PCA
filter_key = 'LAMP1'
filtered_features = list(filter(lambda x: filter_key in x,cell_features))

ordered_cell = cell_df.sort_values(
  by='Time', 
  ascending=True)
ordered_cell['Time']=ordered_cell['Time'].astype("string")
X = ordered_cell[filtered_features]

pca = PCA()
components = pca.fit_transform(X)
labels = {
    str(i): f"PC {i+1} ({var:.1f}%)"
    for i, var in enumerate(pca.explained_variance_ratio_ * 100)
}

fig = px.scatter_matrix(
    components,
    labels=labels,
    dimensions=range(4),
    color=ordered_cell["Time"]
)
fig.update_traces(diagonal_visible=False)
fig.show()

In [ ]:
from sklearn.decomposition import PCA
filter_key = '_MitoTracker'
filtered_features = list(filter(lambda x: filter_key in x,cell_features))

ordered_cell = cell_df.sort_values(
  by='Time', 
  ascending=True)
ordered_cell['Time']=ordered_cell['Time'].astype("string")
X = ordered_cell[filtered_features]

pca = PCA(n_components=3)
components = pca.fit_transform(X)

fig = px.scatter_3d(components, x=0, y=1,z=2, color=ordered_cell['Time'] , labels = {
    str(i): f"PC {i+1} ({var:.1f}%)"
    for i, var in enumerate(pca.explained_variance_ratio_ * 100)
})
fig.show()

In [ ]:
from sklearn.manifold import TSNE
filter_key = 'Entropy_LAMP1'
filtered_features = list(filter(lambda x: filter_key in x,cell_features))

ordered_cell = cell_df.sort_values(
  by='Time', 
  ascending=True)
ordered_cell['Time']=ordered_cell['Time'].astype("string")
X = ordered_cell[filtered_features]

tsne = TSNE(n_components=2, random_state=0)
projections = tsne.fit_transform(X)

fig = px.scatter(
    projections, x=0, y=1,
    color=ordered_cell['Time']
)
fig.show()

In [ ]:
ordered_nuc = combined_nuclei_df.sort_values(
  by='Time', 
  ascending=True)

ordered_nuc['Passage Group'] = ordered_nuc['PassageNumber'].apply(passage_group)

feature = 'AreaShape_Solidity'

fig = px.box(ordered_nuc, x=feature, y='Passage Group', color = 'Passage Group', labels={
                     feature: feature.strip('_'),
                     'Passage Group': 'Passage Group'})

fig.show()